In [ ]:
!pip install kaggle
!pip install transformers
!pip install torch
!pip install opencv-python

In [3]:
from google.colab import files
files.upload()  # This will prompt you to upload the kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"meghaarajeev","key":"b67bca366bc5d2965adc0014abfecc29"}'}

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [5]:
!kaggle datasets list -s ucf-crime


ref                                                          title                                              size  lastUpdated          downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------  ------------------------------------------------  -----  -------------------  -------------  ---------  ---------------  
odins0n/ucf-crime-dataset                                    UCF Crime Dataset                                  11GB  2021-11-11 13:16:24          19294        193  0.875            
mission-ai/crimeucfdataset                                   UCF-Crime                                          33GB  2018-11-11 09:44:56          11552         71  0.3125           
alirakhmaev/ucf-crime-full                                   UCF Crime Full                                     24GB  2021-03-30 19:15:28           1649          9  0.3125           
vigneshwar472/ucaucf-crime-annotation-dataset                UCA(UCF Crime Annotation

In [6]:
!kaggle datasets download -d odins0n/ucf-crime-dataset


Dataset URL: https://www.kaggle.com/datasets/odins0n/ucf-crime-dataset
License(s): CC0-1.0
100% 11.0G/11.0G [01:22<00:00, 244MB/s]
100% 11.0G/11.0G [01:22<00:00, 143MB/s]


In [ ]:
!unzip ucf-crime-dataset.zip -d /content/ucf-crime/


In [19]:
# import cv2
# import os

# video_dir = '/content/ucf-crime/'
# output_dir = '/content/ucf-crime-frames/'
# os.makedirs(output_dir, exist_ok=True)

# frame_rate = 1  # Extract one frame per second

# for video_file in os.listdir(video_dir):
#     video_path = os.path.join(video_dir, video_file)
#     cap = cv2.VideoCapture(video_path)
#     fps = cap.get(cv2.CAP_PROP_FPS)
#     frame_interval = int(fps / frame_rate)
#     frame_count = 0
#     success, frame = cap.read()

#     while success:
#         if frame_count % frame_interval == 0:
#             frame_name = f"{os.path.splitext(video_file)[0]}_frame{frame_count}.jpg"
#             frame_path = os.path.join(output_dir, frame_name)
#             cv2.imwrite(frame_path, frame)
#         success, frame = cap.read()
#         frame_count += 1

#     cap.release()


In [21]:
import cv2
import os

video_dir = '/content/ucf-crime/'
output_dir = '/content/ucf-crime-frames/'
os.makedirs(output_dir, exist_ok=True)

frame_rate = 1  # Extract one frame per second

for video_file in os.listdir(video_dir):
    video_path = os.path.join(video_dir, video_file)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps / frame_rate)
    frame_count = 0
    success, frame = cap.read()

    while success:
        if frame_count % frame_interval == 0:
            # Calculate the timestamp in seconds
            timestamp = frame_count / fps
            # Format the timestamp to two decimal places
            timestamp_formatted = f"{timestamp:.2f}"
            # Construct the frame filename with the timestamp
            frame_name = f"{os.path.splitext(video_file)[0]}_frame{frame_count}_time{timestamp_formatted}s.jpg"
            frame_path = os.path.join(output_dir, frame_name)
            cv2.imwrite(frame_path, frame)
        success, frame = cap.read()
        frame_count += 1

    cap.release()


In [11]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import requests

# Load the processor and model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [29]:
!git clone --no-checkout https://github.com/Xuange923/Surveillance-Video-Understanding.git
%cd Surveillance-Video-Understanding
!git sparse-checkout init --cone
!git sparse-checkout set "UCF Annotation/json"


fatal: destination path 'Surveillance-Video-Understanding' already exists and is not an empty directory.
/content/Surveillance-Video-Understanding
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 692.94 KiB | 9.49 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [15]:
import os
import json


In [37]:
import os
import json

# Define the paths based on the provided folder structure
json_path = '/content/Surveillance-Video-Understanding/UCF Annotation/json/UCFCrime_Train.json'  # Use Train/Test/Val as needed
frames_output_dir = '/content/ucf-crime-frames/'  # Path to extracted frames
frame_annotations_output_dir = '/content/annotatedframes/'  # Directory for frame annotations
frame_annotations_output_path = os.path.join(frame_annotations_output_dir, 'UCFCrime_Frame_Annotations.json')  # Output JSON for frame annotations

# Ensure the output directory exists
os.makedirs(frame_annotations_output_dir, exist_ok=True)

def load_annotations(json_path):
    """
    Load annotations from a JSON file.

    Args:
        json_path (str): Path to the JSON file containing annotations.

    Returns:
        dict: Loaded annotations.
    """
    with open(json_path, 'r') as file:
        annotations = json.load(file)
    return annotations

def align_frames_with_annotations(frames_output_dir, annotations):
    """
    Align frames with annotations.

    Args:
        frames_output_dir (str): Directory containing extracted frames.
        annotations (dict): Annotations data.

    Returns:
        dict: Frame annotations aligned with the frames.
    """
    frame_annotations = {}
    # Implementation to align frames with annotations
    # This is a placeholder; you'll need to implement the actual logic based on your requirements
    return frame_annotations

def save_frame_annotations(frame_annotations, output_path):
    """
    Save frame annotations to a JSON file.

    Args:
        frame_annotations (dict): Frame annotations to save.
        output_path (str): Path to the output JSON file.
    """
    with open(output_path, 'w') as file:
        json.dump(frame_annotations, file, indent=4)

# Load annotations
annotations = load_annotations(json_path)

# Align frames with annotations
frame_annotations = align_frames_with_annotations(frames_output_dir, annotations)

# Save the frame annotations
save_frame_annotations(frame_annotations, frame_annotations_output_path)
